# Book Recommender with a Medallion Data Lake
Bronze (raw) → Silver (clean) → Gold (model-ready) → Item-based collaborative filtering on the Book-Crossing dataset.

## Step 1: Data lake setup and Bronze layer

In [1]:
import os, re, shutil, zipfile, urllib.request
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors

for d in ['raw', 'bronze', 'silver', 'gold']:
    os.makedirs(d, exist_ok=True)

URL = 'https://cdn.freecodecamp.org/project-data/books/book-crossings.zip'
if not os.path.exists('raw/BX-Books.csv'):
    req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open('book-crossings.zip', 'wb') as out:
        shutil.copyfileobj(resp, out)
    with zipfile.ZipFile('book-crossings.zip') as zf:
        zf.extractall('raw')
os.listdir('raw')

['BX-Books.csv', 'BX-Users.csv', 'BX-Book-Ratings.csv']

In [2]:
read = lambda f, **kw: pd.read_csv(f'raw/{f}', sep=';', encoding='latin-1', quotechar='"', escapechar='\\', on_bad_lines='skip', low_memory=False, **kw)

users = read('BX-Users.csv')
books = read('BX-Books.csv', dtype={'ISBN': str, 'Year-Of-Publication': str})
ratings = read('BX-Book-Ratings.csv', dtype={'ISBN': str})

ts = pd.Timestamp.now(tz='UTC')
for name, df in [('users', users), ('books', books), ('ratings', ratings)]:
    df['ingestion_timestamp'] = ts
    df.to_parquet(f'bronze/{name}.parquet', index=False)
os.listdir('bronze')

['books.parquet', 'users.parquet', 'ratings.parquet']

## Step 2: Exploratory data analysis on Bronze

In [3]:
bu, bb, br = (pd.read_parquet(f'bronze/{n}.parquet') for n in ['users', 'books', 'ratings'])
for n, df in [('users', bu), ('books', bb), ('ratings', br)]:
    print(n, df.shape)
    print(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum()}), '\n')

users (278858, 4)
                                   dtype  missing
User-ID                            int64        0
Location                             str        0
Age                              float64   110762
ingestion_timestamp  datetime64[us, UTC]        0 

books (271379, 9)
                                   dtype  missing
ISBN                                 str        0
Book-Title                           str        0
Book-Author                          str        2
Year-Of-Publication                  str        0
Publisher                            str        2
Image-URL-S                          str        0
Image-URL-M                          str        0
Image-URL-L                          str        0
ingestion_timestamp  datetime64[us, UTC]        0 

ratings (1149780, 4)
                                   dtype  missing
User-ID                            int64        0
ISBN                                 str        0
Book-Rating                        int6

**Shape, types, missing values:** Age is mostly missing in `users`, a few `Book-Author`/`Publisher` values are missing in `books`, and `Year-Of-Publication` was read as text, so in Silver I convert it to numeric, impute age, and handle the remaining nulls by dropping rows only when a key is missing.

In [4]:
print(br['Book-Rating'].value_counts().sort_index())
print('share of zeros:', round((br['Book-Rating'] == 0).mean() * 100, 1), '%')

Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64
share of zeros: 62.3 %


**Rating distribution:** zeros are the largest group (implicit feedback, not a real score); in Silver I drop all zeros and keep explicit ratings 1–10.

In [6]:
age = pd.to_numeric(bu['Age'], errors='coerce')
print(age.describe())
print('age <5:', (age < 5).sum(), '| age >100:', (age > 100).sum())
yr = pd.to_numeric(bb['Year-Of-Publication'], errors='coerce')
print(yr.describe())
print('year == 0:', (yr == 0).sum(), '| year > 2026:', (yr > 2026).sum(), '| non-numeric:', yr.isna().sum())

count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: Age, dtype: float64
age <5: 882 | age >100: 366
count    271379.000000
mean       1959.756050
std         258.011363
min           0.000000
25%        1989.000000
50%        1995.000000
75%        2000.000000
max        2050.000000
Name: Year-Of-Publication, dtype: float64
year == 0: 4619 | year > 2026: 11 | non-numeric: 0


**Ages and publication years:** ages include impossible values (0, 1, 244 and so on) and years include 0 and future dates; in Silver I null ages outside 5–100 and impute the median, and null years that are 0 or in the future.

In [7]:
rpu = br.groupby('User-ID').size()
rpb = br.groupby('ISBN').size()
print(rpu.describe(percentiles=[.5, .75, .9, .99]), '\n')
print(rpb.describe(percentiles=[.5, .75, .9, .99]))

count    105283.000000
mean         10.920851
std          90.562825
min           1.000000
50%           1.000000
75%           4.000000
90%          12.000000
99%         174.000000
max       13602.000000
dtype: float64 

count    340556.000000
mean          3.376185
std          12.436252
min           1.000000
50%           1.000000
75%           2.000000
90%           6.000000
99%          36.000000
max        2502.000000
dtype: float64


**Ratings per user and per book:** both are extremely long-tailed (the median user and median book have only a handful of ratings), so Silver applies a minimum-ratings filter to users and books.

In [8]:
n_u, n_b, n_r = br['User-ID'].nunique(), br['ISBN'].nunique(), len(br)
print(f'sparsity: {100 * (1 - n_r / (n_u * n_b)):.4f}%')

sparsity: 99.9968%


**Sparsity:** over 99.99% of user-book pairs have no rating, so Gold stores a sparse matrix and the threshold filter in Silver also helps similarities become meaningful.

## Step 3: Silver layer

In [9]:
su, sb, sr = (bu.drop(columns='ingestion_timestamp').drop_duplicates(),
              bb.drop(columns='ingestion_timestamp').drop_duplicates(),
              br.drop(columns='ingestion_timestamp').drop_duplicates())

su['User-ID'] = pd.to_numeric(su['User-ID'], errors='coerce')
su = su.dropna(subset=['User-ID']).copy()
su['User-ID'] = su['User-ID'].astype(int)

sr['User-ID'] = pd.to_numeric(sr['User-ID'], errors='coerce')
sr['Book-Rating'] = pd.to_numeric(sr['Book-Rating'], errors='coerce')
sr = sr.dropna(subset=['User-ID', 'ISBN', 'Book-Rating']).copy()
sr['User-ID'] = sr['User-ID'].astype(int)
sr = sr[sr['Book-Rating'].between(1, 10)]
# Deduplicate (User-ID, ISBN) pairs so sparse matrix construction does not sum duplicate ratings
sr = sr.sort_values('Book-Rating').drop_duplicates(subset=['User-ID', 'ISBN'], keep='last')

su['Age'] = pd.to_numeric(su['Age'], errors='coerce')
su.loc[(su['Age'] < 5) | (su['Age'] > 100), 'Age'] = np.nan
su['Age'] = su['Age'].fillna(su['Age'].median())

clean = lambda s: s.astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
loc = clean(su['Location']).str.lower().str.split(',', expand=True).apply(lambda col: col.str.strip()).replace('', pd.NA)
su['city'] = loc[0]
su['state'] = loc[1] if 1 in loc else pd.NA
su['country'] = loc.ffill(axis=1).iloc[:, -1].where(loc.notna().sum(axis=1) > 1)
su = su.drop(columns='Location')

sb['Year-Of-Publication'] = pd.to_numeric(sb['Year-Of-Publication'], errors='coerce')
sb.loc[(sb['Year-Of-Publication'] == 0) | (sb['Year-Of-Publication'] > pd.Timestamp.now().year), 'Year-Of-Publication'] = np.nan
sb = sb.drop(columns=[c for c in sb.columns if c.startswith('Image-URL')])
for c in ['Book-Title', 'Book-Author', 'Publisher']:
    sb[c] = clean(sb[c]).str.title()
sb = sb.dropna(subset=['ISBN', 'Book-Title']).drop_duplicates(subset='ISBN')
su = su.drop_duplicates(subset='User-ID')

sr = sr[sr['User-ID'].isin(su['User-ID']) & sr['ISBN'].isin(sb['ISBN'])]
len(su), len(sb), len(sr)

(278858, 271379, 383852)

Choosing the minimum-ratings threshold from the EDA: I check how much data each candidate threshold keeps.

In [10]:
def prune(r, mu, mb):
    while True:
        n = len(r)
        r = r[r.groupby('User-ID')['ISBN'].transform('size') >= mu]
        r = r[r.groupby('ISBN')['User-ID'].transform('size') >= mb]
        if len(r) == n:
            return r

pd.DataFrame([(t, len(x := prune(sr, t, t)), x['User-ID'].nunique(), x['ISBN'].nunique()) for t in [5, 10, 20, 50]],
             columns=['threshold', 'ratings', 'users', 'books']).assign(pct_ratings_kept=lambda d: (100 * d['ratings'] / len(sr)).round(1))

,threshold,ratings,users,books,pct_ratings_kept
0,5,115219,6851,9085,30.0
1,10,41456,1820,2030,10.8
2,20,0,0,0,0.0
3,50,0,0,0,0.0


**Threshold = 10 ratings per user and per book.** The EDA shows the median user and book have very few ratings, so 10 is the smallest value that gives each item enough co-ratings for a stable cosine similarity, while 20 and 50 (see table) discard much more data and shrink the catalogue of recommendable books. The filter is applied repeatedly until no user or book falls below 10.

In [11]:
MIN_RATINGS = 10
sr = prune(sr, MIN_RATINGS, MIN_RATINGS)
su = su[su['User-ID'].isin(sr['User-ID'])]
sb = sb[sb['ISBN'].isin(sr['ISBN'])]
for n, df in [('users', su), ('books', sb), ('ratings', sr)]:
    df.reset_index(drop=True).to_parquet(f'silver/{n}.parquet', index=False)
len(su), len(sb), len(sr)

(1820, 2030, 41456)

## Step 4: Gold layer

In [12]:
su, sb, sr = (pd.read_parquet(f'silver/{n}.parquet') for n in ['users', 'books', 'ratings'])

user_ids = np.sort(sr['User-ID'].unique())
isbns = np.sort(sr['ISBN'].unique())
ui = pd.Series(np.arange(len(user_ids)), index=user_ids)
bi = pd.Series(np.arange(len(isbns)), index=isbns)

matrix = sparse.csr_matrix((sr['Book-Rating'].astype(float), (ui[sr['User-ID']].values, bi[sr['ISBN']].values)), shape=(len(user_ids), len(isbns)))
sparse.save_npz('gold/user_book_matrix.npz', matrix)
pd.DataFrame({'user_idx': ui.values, 'user_id': ui.index}).to_parquet('gold/user_index.parquet', index=False)
pd.DataFrame({'book_idx': bi.values, 'isbn': bi.index}).to_parquet('gold/book_index.parquet', index=False)

book_lookup = sb.rename(columns={'ISBN': 'book_id', 'Book-Title': 'title', 'Book-Author': 'author'})[['book_id', 'title', 'author']]
book_lookup.to_parquet('gold/book_lookup.parquet', index=False)

user_stats = sr.groupby('User-ID')['Book-Rating'].agg(avg_rating='mean', num_ratings='size').reset_index()
book_stats = sr.groupby('ISBN')['Book-Rating'].agg(avg_rating='mean', num_ratings='size').reset_index().rename(columns={'ISBN': 'book_id'})
user_stats.to_parquet('gold/user_stats.parquet', index=False)
book_stats.to_parquet('gold/book_stats.parquet', index=False)
print(matrix.shape, matrix.nnz, f'density: {matrix.nnz / np.prod(matrix.shape):.4%}')

(1820, 2030) 41456 density: 1.1221%


## Step 5: Item-based collaborative filtering

In [13]:
matrix = sparse.load_npz('gold/user_book_matrix.npz')
user_index = pd.read_parquet('gold/user_index.parquet').set_index('user_id')['user_idx']
book_index = pd.read_parquet('gold/book_index.parquet')
lookup = pd.read_parquet('gold/book_lookup.parquet').set_index('book_id')
stats = pd.read_parquet('gold/book_stats.parquet').set_index('book_id')

item_user = matrix.T.tocsr()
isbn_of = book_index.sort_values('book_idx')['isbn'].values
meta = lookup.loc[isbn_of].reset_index()
meta['num_ratings'] = stats.loc[isbn_of, 'num_ratings'].values

knn = NearestNeighbors(metric='cosine', algorithm='brute').fit(item_user)
item_user.shape

(2030, 1820)

In [14]:
def similar_books(title, n=10):
    q = re.escape(title.strip().lower())
    hit = meta[meta['title'].str.lower().eq(title.strip().lower()).fillna(False)]
    if hit.empty:
        hit = meta[meta['title'].str.lower().str.contains(q, na=False)]
    if hit.empty:
        raise ValueError(f'No book found matching: {title}')
    i = hit.sort_values('num_ratings', ascending=False).index[0]
    k_nn = min(n + 1, item_user.shape[0])
    dist, idx = knn.kneighbors(item_user[i], n_neighbors=k_nn)
    out = meta.iloc[idx[0][1:]][['title', 'author']].copy()
    out['similarity'] = (1 - dist[0][1:]).round(3)
    print('Query:', meta.loc[i, 'title'])
    return out.reset_index(drop=True)

In [15]:
def recommend(user_id, n=10, like_threshold=7, k=50):
    if user_id not in user_index.index:
        raise KeyError(f'Unknown user_id: {user_id}')
    u = user_index[user_id]
    row = matrix[u]
    read, scores = row.indices, row.data
    if len(read) == 0:
        return pd.DataFrame(columns=['title', 'author', 'score'])
    liked = read[scores >= like_threshold]
    if len(liked) == 0:
        liked = read[np.argsort(-scores)[:5]]
    w = dict(zip(read, scores))
    k_nn = min(k + 1, item_user.shape[0])
    dist, idx = knn.kneighbors(item_user[liked], n_neighbors=k_nn)
    total = {}
    for b, d_row, i_row in zip(liked, dist, idx):
        for d, j in zip(d_row, i_row):
            if j != b and j not in w:
                total[j] = total.get(j, 0.0) + max(0.0, 1.0 - float(d)) * float(w[b])
    top = sorted(total, key=total.get, reverse=True)[:n]
    out = meta.iloc[top][['title', 'author']].copy()
    out['score'] = [round(total[j], 3) for j in top]
    return out.reset_index(drop=True)

In [16]:
similar_books(meta.sort_values('num_ratings').iloc[-1]['title'])

Query: The Lovely Bones: A Novel


,title,author,similarity
0,Where The Heart Is (Oprah'S Book Club (Paperba...,Billie Letts,0.219
1,Lucky : A Memoir,Alice Sebold,0.218
2,The Da Vinci Code,Dan Brown,0.178
3,The Pilot'S Wife : A Novel,Anita Shreve,0.176
4,Good In Bed,Jennifer Weiner,0.176
5,Nights In Rodanthe,Nicholas Sparks,0.170
6,The Pact: A Love Story,Jodi Picoult,0.168
7,The Lake House,James Patterson,0.166
8,Lucky,Alice Sebold,0.165
9,White Oleander : A Novel (Oprah'S Book Club),Janet Fitch,0.154


## Step 6: Demo and sanity check

In [17]:
rng = np.random.default_rng(42)
ustats = pd.read_parquet('gold/user_stats.parquet')
demo_users = rng.choice(ustats.loc[ustats['num_ratings'].between(20, 100), 'User-ID'].values, 3, replace=False)

def show(uid):
    row = matrix[user_index[uid]]
    liked = pd.DataFrame({'idx': row.indices, 'rating': row.data}).sort_values('rating', ascending=False).head(8)
    liked = pd.concat([meta.iloc[liked['idx']][['title', 'author']].reset_index(drop=True), liked['rating'].reset_index(drop=True)], axis=1)
    print(f'=== User {uid} | {row.nnz} ratings ===\nHighly rated:')
    print(liked.to_string(index=False))
    print('\nRecommended:')
    print(recommend(uid).to_string(index=False), '\n')

for u in demo_users:
    show(u)

=== User 187863 | 24 ratings ===
Highly rated:
                              title             author  rating
The Red Tent (Bestselling Backlist)      Anita Diamant    10.0
                  Midwives: A Novel    Chris Bohjalian     9.0
    One Flew Over The Cuckoo'S Nest          Ken Kesey     9.0
    Angela'S Ashes (Mmp) : A Memoir      Frank Mccourt     9.0
                       Bag Of Bones       Stephen King     9.0
        Loves Music, Loves To Dance Mary Higgins Clark     9.0
                      The Dark Half       Stephen King     9.0
              River, Cross My Heart      Breena Clarke     8.0

Recommended:
                                       title            author  score
                                  The Reader  Bernhard Schlink  7.556
                           Dolores Claiborne      Stephen King  7.490
                               Gerald'S Game      Stephen King  7.463
                       The Rapture Of Canaan    Sheri Reynolds  7.213
                      

**Sanity check:** the recommended titles should share genre or author with the books each user rated highly, and none of them should appear in the user's own list, since `recommend()` removes already-read books.

## Step 7: Offline evaluation and baseline comparison

To quantify recommendation quality without data leakage, we perform a per-user 80/20 train/test split on the Gold interaction matrix, fit `NearestNeighbors` strictly on the training matrix, and evaluate top-`K` (`K = 10`) recommendations against a non-personalized **Popularity** baseline on held-out relevant books (`Book-Rating >= 7`).

In [18]:
def train_test_split_csr(mat, test_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    tr_rows, tr_cols, tr_vals = [], [], []
    te_relevant = {}
    for u in range(mat.shape[0]):
        row = mat[u]
        cols, vals = row.indices, row.data
        n_test = max(1, int(round(len(cols) * test_frac)))
        perm = rng.permutation(len(cols))
        te_idx, tr_idx = perm[:n_test], perm[n_test:]
        tr_rows.extend([u] * len(tr_idx))
        tr_cols.extend(cols[tr_idx])
        tr_vals.extend(vals[tr_idx])
        rel = set(cols[te_idx][vals[te_idx] >= 7])
        if rel:
            te_relevant[u] = rel
    train_mat = sparse.csr_matrix((tr_vals, (tr_rows, tr_cols)), shape=mat.shape)
    return train_mat, te_relevant

train_mat, test_rel = train_test_split_csr(matrix, test_frac=0.2, seed=42)
train_item_user = train_mat.T.tocsr()
eval_knn = NearestNeighbors(metric='cosine', algorithm='brute').fit(train_item_user)

# Popularity ranking from training interactions only
pop_order = np.argsort(-np.asarray(train_mat.getnnz(axis=0)).ravel())

def eval_recommend_cf(u, K=10, like_threshold=7, k_neighbors=50):
    row = train_mat[u]
    read, scores = row.indices, row.data
    if len(read) == 0:
        return []
    liked = read[scores >= like_threshold]
    if len(liked) == 0:
        liked = read[np.argsort(-scores)[:5]]
    seen = set(read)
    w = dict(zip(read, scores))
    dist, idx = eval_knn.kneighbors(train_item_user[liked], n_neighbors=min(k_neighbors + 1, train_item_user.shape[0]))
    total = {}
    for b, d_row, i_row in zip(liked, dist, idx):
        for d, j in zip(d_row, i_row):
            if j != b and j not in seen:
                total[j] = total.get(j, 0.0) + max(0.0, 1.0 - float(d)) * float(w[b])
    return sorted(total, key=total.get, reverse=True)[:K]

def eval_recommend_pop(u, K=10):
    seen = set(train_mat[u].indices)
    recs = []
    for j in pop_order:
        if j not in seen:
            recs.append(int(j))
            if len(recs) == K:
                break
    return recs

def evaluate_model(rec_fn, users, K=10):
    precisions, recalls, hits, ndcgs = [], [], [], []
    recommended_items = set()
    idcg_cache = {m: sum(1.0 / np.log2(r + 2) for r in range(min(m, K))) for m in range(1, K + 1)}
    for u in users:
        rel = test_rel[u]
        recs = rec_fn(u, K=K)
        recommended_items.update(recs)
        hits_u = [1 if item in rel else 0 for item in recs]
        n_hit = sum(hits_u)
        precisions.append(n_hit / K)
        recalls.append(n_hit / len(rel))
        hits.append(1.0 if n_hit > 0 else 0.0)
        dcg = sum(h / np.log2(rank + 2) for rank, h in enumerate(hits_u))
        idcg = idcg_cache.get(min(len(rel), K), sum(1.0 / np.log2(r + 2) for r in range(min(len(rel), K))))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return {
        f'Precision@{K}': round(float(np.mean(precisions)), 4),
        f'Recall@{K}': round(float(np.mean(recalls)), 4),
        f'HitRate@{K}': round(float(np.mean(hits)), 4),
        f'NDCG@{K}': round(float(np.mean(ndcgs)), 4),
        f'CatalogCoverage@{K}': round(len(recommended_items) / matrix.shape[1], 4),
    }

eval_users = sorted(test_rel.keys())
results = pd.DataFrame([
    {'Model': 'Item-Based CF (Cosine, k=50)', **evaluate_model(eval_recommend_cf, eval_users, K=10)},
    {'Model': 'Popularity Baseline', **evaluate_model(eval_recommend_pop, eval_users, K=10)},
])
print(f'Evaluated on {len(eval_users)} users with >=1 relevant held-out book:')
results

Evaluated on 1766 users with >=1 relevant held-out book:


,Model,Precision@10,Recall@10,HitRate@10,NDCG@10,CatalogCoverage@10
0,"Item-Based CF (Cosine, k=50)",0.0404,0.1143,0.2888,0.0974,0.7985
1,Popularity Baseline,0.0152,0.0409,0.1365,0.0322,0.0094


**Evaluation takeaway:** Item-Based CF outperforms the popularity baseline on personalized ranking metrics (Precision@10, Recall@10, HitRate@10, NDCG@10) while achieving substantially higher catalog coverage—recommending niche and domain-relevant books rather than repeatedly surfacing the same global bestsellers.